# Apollo 15 HFE Validation

Compare the Lunar-V2 1-D thermal solver against the Apollo 15 Heat Flow Experiment (HFE)
temperature timeseries (Nagihara et al. 2018, PDS bundle
`urn:nasa:pds:a15_17_hfe_concatenated`).

**Apollo 15 site**: 26.13 °N, 3.63 °E (Mare Imbrium, 1971-2011 continuous record).

Sensors are buried at depths from ~35 cm to ~139 cm. At those depths the diurnal
thermal wave is already strongly attenuated; the dominant signal is a slow secular
warming (~2 K over the mission life) and the vertical geothermal gradient.
This notebook checks that the solver reproduces:

1. Roughly correct mean temperatures at depth (240–260 K range).
2. Correct depth ordering: deeper sensors are cooler (positive geothermal gradient
   from below means the deep interior is hotter than the cold mean surface, so
   subsurface T *increases* with depth toward equilibrium).
3. Attenuated diurnal amplitude vs the analytical skin-depth prediction.

**Requirements**: `data/apollo/` populated (see `data/README.md`).

In [ ]:
from __future__ import annotations
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import matplotlib
matplotlib.use('Agg')  # remove if running in Jupyter with display
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from lunar.validation import load_apollo_hfe_temperature, load_apollo_hfe_depth
from lunar.grid import make_geometric_grid
from lunar.properties import conductivity_hayne, density_hayne, specific_heat
from lunar.constants import Q_B_EQUATORIAL, SIGMA_SB, EMISSIVITY_DEFAULT
from lunar.solver import PixelInputs, solve_pixel

print('Imports OK')

## 1  Load the Apollo 15 HFE data

In [ ]:
# Load three fast-sensor channels from Probe 1 and the depth table.
rec_f1 = load_apollo_hfe_temperature('a15', 'p1f1')
rec_f2 = load_apollo_hfe_temperature('a15', 'p1f2')
rec_f3 = load_apollo_hfe_temperature('a15', 'p1f3')
depth_table = load_apollo_hfe_depth('a15', 1)

print(f"p1f1: {rec_f1.T.size:,} samples,  T ∈ [{rec_f1.T.min():.1f}, {rec_f1.T.max():.1f}] K")
print(f"p1f2: {rec_f2.T.size:,} samples,  T ∈ [{rec_f2.T.min():.1f}, {rec_f2.T.max():.1f}] K")
print(f"p1f3: {rec_f3.T.size:,} samples,  T ∈ [{rec_f3.T.min():.1f}, {rec_f3.T.max():.1f}] K")

unique_depths_cm = np.unique(depth_table['depth_cm'])
print(f"\nSensor depths [cm]: {unique_depths_cm}")

In [ ]:
# Convert sensor-clock to fractional years for plotting.
# The PDS archive uses an internal clock; we only need relative time.
# The first measurement is ~1971-07-31 (Apollo 15 landed 1971-07-30).
# Approximate: t0 = rec_f1.time_s[0] maps to day 0.

SECONDS_PER_YEAR = 365.25 * 86400.0

def to_relative_years(time_s, t0):
    return (time_s - t0) / SECONDS_PER_YEAR

t0 = rec_f1.time_s[0]
yr_f1 = to_relative_years(rec_f1.time_s, t0)
yr_f2 = to_relative_years(rec_f2.time_s, t0)
yr_f3 = to_relative_years(rec_f3.time_s, t0)

print(f"Record spans {yr_f1[-1]:.1f} years after initial deployment")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

for ax, rec, yr, label, depth in zip(
    axes,
    [rec_f1, rec_f2, rec_f3],
    [yr_f1, yr_f2, yr_f3],
    ['p1f1 (~35 cm)', 'p1f2 (~45 cm)', 'p1f3 (~73 cm)'],
    [35, 45, 73],
):
    ax.plot(yr, rec.T, lw=0.4, color='steelblue', alpha=0.8)
    ax.set_ylabel('T [K]', fontsize=9)
    ax.set_title(f'Apollo 15 HFE — Probe 1 {label}', fontsize=10)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f'))

axes[-1].set_xlabel('Years after deployment', fontsize=10)
fig.tight_layout()
fig.savefig('../data/apollo/a15_p1_timeseries.png', dpi=150)
print('Saved a15_p1_timeseries.png')
plt.close(fig)

## 2  Mean temperature profile vs depth

The geothermal gradient at Apollo 15 is well-constrained: ~1.8 K/m
(Langseth et al. 1976). At ~35–140 cm depth the temperature profile
is nearly linear, sitting in the range 250–260 K.

In [ ]:
# Assemble (depth_cm, mean_T) pairs from all probe 1 fast sensors.
probes = [
    ('p1f1', 35),
    ('p1f2', 45),
    ('p1f3', 73),
    ('p1f4', 84),
]

depth_cm_obs = []
T_mean_obs   = []
T_std_obs    = []

for probe, depth in probes:
    try:
        rec = load_apollo_hfe_temperature('a15', probe)
        # Skip obviously flagged samples.
        good = rec.flags == 0
        depth_cm_obs.append(depth)
        T_mean_obs.append(float(rec.T[good].mean()))
        T_std_obs.append(float(rec.T[good].std()))
        print(f'{probe} (depth={depth} cm):  T_mean = {rec.T[good].mean():.2f} K,  σ = {rec.T[good].std():.3f} K')
    except FileNotFoundError as e:
        print(f'  SKIP: {e}')

depth_cm_obs = np.array(depth_cm_obs)
T_mean_obs   = np.array(T_mean_obs)
T_std_obs    = np.array(T_std_obs)

## 3  Run the 1-D solver at Apollo 15 coordinates

We drive the surface with a simplified mean-daily sinusoidal insolation
representing a mid-latitude lunar surface at ~26 °N. Peak insolation is
`S0 * cos(26°) ≈ 1220 W m⁻²`; the diurnal average is half that.

A 10-lunation spin-up is applied; the converged temperature profile is
then compared against the HFE measurements.

In [ ]:
# Apollo 15 site parameters
LAT_A15 = 26.13  # deg N
ALBEDO   = 0.12  # Hayne 2017 nominal
EMIS     = EMISSIVITY_DEFAULT  # 0.95

# Lunar sidereal period ≈ 27.32 days
T_LUNAR   = 27.321661 * 86400.0   # s
S0        = 1361.0                 # W m^-2 solar constant
cos_lat   = np.cos(np.deg2rad(LAT_A15))

# One lunation, 3600-s time steps
dt      = 3600.0                   # s
N_t     = int(T_LUNAR / dt) + 1
t_s     = np.linspace(0.0, T_LUNAR, N_t)

# Sinusoidal insolation: S(t) = S0*cos_lat*max(0, cos(2π t/T_LUNAR))
# (models sun crossing horizon twice per lunation at the equator;
# at 26 N the sub-solar latitude keeps direct illumination for most of the day)
phase      = 2.0 * np.pi * t_s / T_LUNAR
insolation = S0 * cos_lat * np.maximum(0.0, np.cos(phase))

# Geometric grid — diurnal skin depth at 26 N ≈ 0.05 m
grid = make_geometric_grid(z_max=5.0, dz0=0.005, growth=1.08)
print(f'Grid: {grid.n_layers} layers,  z_max = {grid.z_mid[-1]:.2f} m')

# Solver inputs
inputs = PixelInputs(
    grid=grid,
    t=t_s,
    bc_mode='radiative',
    insolation=insolation,
    albedo=ALBEDO,
    emissivity=EMIS,
    Q_b=Q_B_EQUATORIAL,          # 0.018 W m^-2 (equatorial)
    n_lunations_spinup=10,
    spinup_tol_K=0.01,
)

print('Running solver (10-lunation spin-up)...')
out = solve_pixel(inputs)
print(f'  Spin-up: {out.n_spinup_cycles} cycles,  converged={out.converged}')

In [ ]:
# Time-mean temperature profile from the last lunation
T_mean_model = out.T.mean(axis=1)   # shape (N_z,)
z_m = out.z                          # depths [m]

fig, ax = plt.subplots(figsize=(6, 8))

# Model profile
ax.plot(T_mean_model, z_m * 100.0, '-', color='tomato', lw=2.0, label='Solver (mean)')

# Apollo 15 HFE observations
ax.errorbar(
    T_mean_obs, depth_cm_obs,
    xerr=T_std_obs,
    fmt='o', color='steelblue', capsize=4, ms=7,
    label='Apollo 15 HFE p1',
)

ax.invert_yaxis()
ax.set_xlabel('Mean temperature [K]', fontsize=12)
ax.set_ylabel('Depth [cm]', fontsize=12)
ax.set_title('Apollo 15 — subsurface mean-T profile', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig('../data/apollo/a15_mean_T_profile.png', dpi=150)
print('Saved a15_mean_T_profile.png')
plt.close(fig)

## 4  Diurnal amplitude vs depth — skin-depth check

For a sinusoidal surface forcing the thermal wave amplitude decays as

$$A(z) = A_0 \exp(-z / l_d),  \qquad l_d = \sqrt{\frac{K}{\rho c_p \omega}}$$

where $\omega = 2\pi / T_{\text{lunar}}$. For Hayne (2017) nominal properties
at the Apollo 15 site $l_d \approx 0.04$–$0.06$ m, so the diurnal signal
should be < 1 K at 35 cm depth — consistent with the HFE σ values.

In [ ]:
# Diurnal amplitude: half peak-to-peak over the last lunation
T_amplitude = 0.5 * (out.T.max(axis=1) - out.T.min(axis=1))

# Analytical skin-depth estimate at surface density
from lunar.properties import conductivity_hayne, density_hayne, specific_heat

T_ref   = T_mean_model[0]  # surface temperature for property evaluation
rho0    = float(density_hayne(np.array([0.0]))[0])
K0      = float(conductivity_hayne(np.array([300.0]), np.array([0.0]))[0])
cp0     = float(specific_heat(np.array([T_ref]))[0])
omega   = 2.0 * np.pi / T_LUNAR
l_d     = np.sqrt(K0 / (rho0 * cp0 * omega))
print(f'Hayne nominal skin depth l_d = {l_d*100:.2f} cm')

A0_model = T_amplitude[0]
z_analytical = np.linspace(0, 0.8, 500)
A_analytical = A0_model * np.exp(-z_analytical / l_d)

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(z_m * 100.0, T_amplitude, '-', color='tomato', lw=2, label='Solver')
ax.semilogy(z_analytical * 100.0, A_analytical, '--', color='gray', lw=1.5,
            label=f'Analytical ($l_d$={l_d*100:.1f} cm)')

# Apollo 15 HFE σ as approximate amplitude
ax.scatter(depth_cm_obs, T_std_obs, s=60, zorder=5, color='steelblue',
           label='Apollo 15 HFE σ')

ax.set_xlabel('Depth [cm]', fontsize=12)
ax.set_ylabel('Diurnal amplitude [K]', fontsize=12)
ax.set_title('Thermal wave amplitude vs depth', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, which='both', alpha=0.3)
ax.set_xlim(0, 80)

fig.tight_layout()
fig.savefig('../data/apollo/a15_amplitude_vs_depth.png', dpi=150)
print('Saved a15_amplitude_vs_depth.png')
plt.close(fig)

## 5  Residual summary

In [ ]:
# Interpolate model mean-T onto the HFE sensor depths.
T_model_at_hfe = np.interp(depth_cm_obs / 100.0, z_m, T_mean_model)
residuals       = T_model_at_hfe - T_mean_obs
rms             = float(np.sqrt(np.mean(residuals**2)))

print('Depth [cm] | HFE mean T [K] | Model mean T [K] | Residual [K]')
print('-' * 60)
for d, T_obs, T_mod, res in zip(depth_cm_obs, T_mean_obs, T_model_at_hfe, residuals):
    print(f'  {d:5.0f}    |   {T_obs:.2f}        |   {T_mod:.2f}           |  {res:+.2f}')
print('-' * 60)
print(f'RMS residual: {rms:.2f} K')
print()
if rms < 5.0:
    print('PASS — RMS < 5 K (expected for sinusoidal proxy insolation)')
else:
    print('NOTE — RMS > 5 K; may need real SPICE-driven insolation (see notebook 03)')